<a href="https://colab.research.google.com/github/jinsujini/SSWU_AI_TEAM4/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 스켈레톤 이미지 기반 CNN과 LSTM 필라테스 동작 인식 및 정확도 분석 모델


**라이브러리 import / 드라이브 마운트**


In [123]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import os

import glob
import random
import imageio
from IPython.display import display, Image as IPImage
import io

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [124]:
dataset_path = '/content/drive/MyDrive/datasets/SSWU_AI_TEAM4-data_soop/skeleton_images'
action_name = 'Swimming'

train_videos, val_videos, test_videos = split_videos_from_action(dataset_path, action_name)

# ✅ 증강용 transform 정의
augmented_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor()
])

# ✅ 평가용 기본 transform도 같이 정의
base_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor()
])


In [125]:
# ✅ 1. 영상 폴더 이름 기준 train/val/test 나누기
def split_videos_from_action(root_dir, action_name='Swimming', train_ratio=0.5, val_ratio=0.2, test_ratio=0.3, seed=42):
    action_path = os.path.join(root_dir, action_name)
    video_folders = [f for f in os.listdir(action_path) if os.path.isdir(os.path.join(action_path, f))]
    random.seed(seed)
    random.shuffle(video_folders)
    total = len(video_folders)
    n_train = int(total * train_ratio)
    n_val = int(total * val_ratio)
    return video_folders[:n_train], video_folders[n_train:n_train+n_val], video_folders[n_train+n_val:]

In [126]:

class FrameSequenceDataset(Dataset):
    def __init__(self, root_dir, action_name, video_list, seq_len=120, stride=30, transform=None):
        self.samples = []
        self.seq_len = seq_len
        self.stride = stride
        self.transform = transform
        action_path = os.path.join(root_dir, action_name)

        for video_folder in video_list:
            video_path = os.path.join(action_path, video_folder)
            if not os.path.isdir(video_path):
                continue

            if 'wrong' in video_folder:
                img_dir = os.path.join(video_path, 'skeleton_images')
                label_path = os.path.join(video_path, 'labels.json')
                if not os.path.exists(label_path):
                    continue
                with open(label_path, 'r') as f:
                    label_info = json.load(f)
                image_paths = sorted(glob.glob(os.path.join(img_dir, '**', '*.png'), recursive=True))
            else:
                image_paths = sorted(glob.glob(os.path.join(video_path, '*.png')))
                label_path = None
                label_info = None

            total_frames = len(image_paths)
            if total_frames < self.seq_len:
                continue

            # 슬라이딩 윈도우 시퀀스 생성
            for start in range(0, total_frames - self.seq_len + 1, self.stride):
                end = start + self.seq_len
                frames = image_paths[start:end]
                if label_info:
                    frame_labels = [item['is_correct'] for item in label_info[start:end]]
                    label = 0 if any(l == 0 for l in frame_labels) else 1
                else:
                    label = 1

                self.samples.append((frames, label, video_folder, label_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_paths, label, video_folder, label_path = self.samples[idx]

        imgs = []
        for path in frame_paths:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            imgs.append(img)

        joint_vector = torch.zeros(len(joint_names))

        if 'wrong' in video_folder and label_path and os.path.exists(label_path):
            with open(label_path, 'r') as f:
                label_info = json.load(f)
            for i in range(len(frame_paths)):
                if i < len(label_info):
                    for jname in label_info[i].get("wrong_joints", []):
                        if jname in joint_names:
                            joint_vector[joint_names.index(jname)] = 1

        return torch.stack(imgs), torch.tensor(label, dtype=torch.float32), joint_vector


In [127]:
wrong_samples = [s for s in train_dataset.samples if s[1] == 0]
train_dataset.samples += wrong_samples * 8

In [128]:
# ✅ 3. 모델 정의 (CNN + LSTM)
class CNNLSTMClassifier(nn.Module):
    def __init__(self, cnn_out_dim=128, hidden_dim=64, num_layers=1, num_joints=12):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, cnn_out_dim, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.lstm = nn.LSTM(cnn_out_dim, hidden_dim, num_layers, batch_first=True)

        self.correct_fc = nn.Linear(hidden_dim, 1)
        self.joint_fc   = nn.Linear(hidden_dim, num_joints)

    def forward(self, x):  # x: (B, T, C, H, W)
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)
        feats = self.cnn(x).view(B, T, -1)
        _, (hn, _) = self.lstm(feats)

        last_hidden = hn[-1]

        is_correct_prob = torch.sigmoid(self.correct_fc(last_hidden))
        joint_probs     = torch.sigmoid(self.joint_fc(last_hidden))

        return is_correct_prob.squeeze(1), joint_probs

joint_names = [
    "Head", "Neck", "LShoulder", "LElbow", "LWrist",
    "RShoulder", "RElbow", "RWrist", "Hip", "LKnee", "RKnee", "Ankle"
]

def decode_output(is_correct_probs, joint_probs, threshold=0.5):
    results = []
    for i in range(len(is_correct_probs)):
        prob = is_correct_probs[i].item()
        joints = [
            joint_names[j] for j, p in enumerate(joint_probs[i]) if p.item() > threshold
        ]
        results.append({
            "is_correct_prob": round(prob, 4),
            "wrong_joints": joints
        })
    return results

In [129]:
# ✅ 학습 함수 (BCELoss 기반)
def train_model(model, train_loader, val_loader, device, epochs=5, lambda_joint=1.0):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_correct = nn.BCELoss()
    loss_joint = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct_total = 0
        total_samples = 0

        print(f"\n🔁 Epoch {epoch+1} 시작:")
        for x, y_correct, y_joint in tqdm(train_loader, desc=f"Train Epoch {epoch+1}"):
            x = x.to(device)
            y_correct = y_correct.to(device)
            y_joint = y_joint.to(device)

            pred_correct, pred_joint = model(x)
            loss1 = loss_correct(pred_correct, y_correct)
            loss2 = loss_joint(pred_joint, y_joint)
            loss = loss1 + lambda_joint * loss2

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            preds = (pred_correct > 0.5).float()
            correct_total += (preds == y_correct).sum().item()
            total_samples += y_correct.size(0)

        acc = correct_total / total_samples * 100
        print(f"[Epoch {epoch+1}] Train Loss: {total_loss:.4f} | Accuracy: {acc:.2f}%")

        # ✅ 검증 루프
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for x, y_correct, y_joint in tqdm(val_loader, desc=f"Validation {epoch+1}"):
                x = x.to(device)
                y_correct = y_correct.to(device)
                y_joint = y_joint.to(device)

                pred_correct, pred_joint = model(x)
                loss1 = loss_correct(pred_correct, y_correct)
                loss2 = loss_joint(pred_joint, y_joint)
                loss = loss1 + lambda_joint * loss2
                val_loss += loss.item()

                preds = (pred_correct > 0.5).float()
                val_correct += (preds == y_correct).sum().item()
                val_total += y_correct.size(0)

        val_acc = val_correct / val_total * 100
        print(f"            → Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n")


In [130]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNLSTMClassifier(num_joints=len(joint_names)).to(device)
train_dataset = FrameSequenceDataset(
    dataset_path,
    action_name,
    train_videos,
    seq_len=120,
    stride=30,
    transform=augmented_transform
)

val_dataset = FrameSequenceDataset(
    dataset_path,
    action_name,
    val_videos,
    transform=base_transform
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)



In [ ]:
train_model(
    model,
    train_loader,
    val_loader,
    device=device,
    epochs=5,           # ← 조절 가능
    lambda_joint=2.0    # ← 관절 손실 가중치 (0.5 ~ 2.0 추천)
)



🔁 Epoch 1 시작:


Train Epoch 1:  75%|███████▌  | 6/8 [03:10<01:03, 31.61s/it]

In [ ]:
from collections import Counter

val_labels = [int(label) for _, label, _, _ in val_dataset.samples]
print("✅ Validation 정오 분포:", Counter(val_labels))



In [ ]:
# ✅ test용 transform은 보통 기본 transform과 동일
test_dataset = FrameSequenceDataset(
    dataset_path,
    action_name,
    test_videos,
    transform=base_transform
)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [ ]:
model.eval()
from pprint import pprint  # 보기 좋게 출력하려고

with torch.no_grad():
    for x, y_correct, y_joint in test_loader:
        x = x.to(device)
        pred_correct, pred_joint = model(x)
        results = decode_output(pred_correct.cpu(), pred_joint.cpu())
        pprint(results)
        break  # 🔁 첫 배치만 확인해보자


In [ ]:
for i in range(3):
    _, label, joint_vec = train_dataset[i]
    print(f"Label: {label}, Joint Vector: {joint_vec.tolist()}")


In [ ]:
wrong_count = sum(1 for _, _, vname, _ in train_dataset.samples if 'wrong' in vname)
print(f"학습셋 내 wrong 시퀀스 수: {wrong_count}")


In [ ]:
label_0_count = sum(1 for _, label, _, _ in train_dataset.samples if label == 0)
print(f"학습셋 내 label 0 (틀림) 수: {label_0_count}")
